# Proyek Akhir: Menyelesaikan Permasalahan Institusi Pendidikan

- **Nama:** Muhammad Farhan Zahid
- **Email:** farhanzahidmuhammad@gmail.com

## Persiapan

### Import Library

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

import joblib

# Styling
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
print('Libraries loaded successfully.')

### Load Dataset

In [ ]:
DATA_URL = (
    'https://raw.githubusercontent.com/dicodingacademy/dicoding_dataset'
    '/main/students_performance/data.csv'
)

df_raw = pd.read_csv(DATA_URL, sep=';')
print(f'Raw dataset shape : {df_raw.shape}')
print(f'Columns           : {list(df_raw.columns)}')
print(f'Status values     : {df_raw["Status"].value_counts().to_dict()}')
df_raw.head()

## Data Understanding

In [ ]:
print('=== Dataset Info ===')
print(f'Rows    : {df_raw.shape[0]}')
print(f'Columns : {df_raw.shape[1]}')
print()
df_raw.info()

In [ ]:
print('=== Descriptive Statistics ===')
df_raw.describe().T

In [ ]:
print('=== Missing Values ===')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
result = missing_df[missing_df['Missing Count'] > 0]
print(result.to_string() if len(result) > 0 else 'No missing values found.')

print('\n=== Duplicate Rows ===')
print(f'Duplicate rows: {df_raw.duplicated().sum()}')

In [ ]:
# Distribusi Status (target)
status_counts = df_raw['Status'].value_counts()
status_pct = df_raw['Status'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#F44336', '#4CAF50', '#2196F3']
axes[0].bar(status_counts.index, status_counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, v in enumerate(status_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')
axes[0].set_title('Distribusi Status Siswa', fontweight='bold')
axes[0].set_ylabel('Jumlah Siswa')

axes[1].pie(
    status_pct.values,
    labels=[f'{s}\n({p:.1f}%)' for s, p in zip(status_pct.index, status_pct.values)],
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Proporsi Status Siswa', fontweight='bold')

plt.tight_layout()
plt.show()
print(f'\nDistribusi Status:')
print(status_counts.to_string())

In [ ]:
# EDA - Faktor Finansial vs Dropout
df_eda = df_raw.copy()

financial_cols = ['Tuition_fees_up_to_date', 'Debtor', 'Scholarship_holder']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, financial_cols):
    ct = pd.crosstab(df_eda[col], df_eda['Status'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#F44336', '#4CAF50', '#2196F3'], edgecolor='white')
    ax.set_title(f'{col}\nvs Status', fontweight='bold')
    ax.set_ylabel('Persentase (%)')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(loc='upper right', fontsize=8)

plt.suptitle('Faktor Finansial vs Status Siswa', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# EDA - Faktor Demografis vs Dropout
demo_cols = ['Gender', 'Marital_status', 'International', 'Displaced']
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, col in zip(axes, demo_cols):
    ct = pd.crosstab(df_eda[col], df_eda['Status'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#F44336', '#4CAF50', '#2196F3'], edgecolor='white')
    ax.set_title(f'{col}\nvs Status', fontweight='bold')
    ax.set_ylabel('Persentase (%)')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(fontsize=8)

plt.suptitle('Faktor Demografis vs Status Siswa', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# EDA - Performa Akademik vs Status
academic_cols = [
    'Curricular_units_1st_sem_approved', 'Curricular_units_1st_sem_grade',
    'Curricular_units_2nd_sem_approved', 'Curricular_units_2nd_sem_grade',
    'Age_at_enrollment', 'Admission_grade'
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

colors_map = {'Dropout': '#F44336', 'Graduate': '#4CAF50', 'Enrolled': '#2196F3'}

for i, col in enumerate(academic_cols):
    for status, color in colors_map.items():
        data = df_eda[df_eda['Status'] == status][col]
        axes[i].hist(data, bins=20, alpha=0.55, color=color, label=status, edgecolor='none')
    axes[i].set_title(f'Distribusi {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=8)

plt.suptitle('Performa Akademik & Usia vs Status Siswa', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot Usia Pendaftaran per Status
fig, ax = plt.subplots(figsize=(8, 5))
order = ['Dropout', 'Enrolled', 'Graduate']
colors_box = ['#F44336', '#2196F3', '#4CAF50']

for i, (status, color) in enumerate(zip(order, colors_box)):
    data = df_eda[df_eda['Status'] == status]['Age_at_enrollment']
    bp = ax.boxplot(data, positions=[i], widths=0.5, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=2))

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(order)
ax.set_title('Distribusi Usia Saat Mendaftar per Status', fontweight='bold')
ax.set_ylabel('Usia')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numerical only)
num_df = df_raw.select_dtypes(include=[np.number])
corr_matrix = num_df.corr()

plt.figure(figsize=(18, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdYlGn',
            center=0, linewidths=0.3, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap - Fitur Numerik', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## Data Preparation

In [ ]:
df = df_raw.copy()

# Drop baris dengan Status null
df = df.dropna(subset=['Status'])

# Fill null pada kolom lainnya
for col in df.columns:
    if df[col].isnull().any():
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

print(f'Shape setelah cleaning: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Status distribution:\n{df["Status"].value_counts()}')

In [ ]:
# Export data untuk Business Dashboard
df_dashboard = df.copy()
df_dashboard['Is_Dropout'] = (df_dashboard['Status'] == 'Dropout').astype(int)
df_dashboard['Age_Group'] = pd.cut(
    df_dashboard['Age_at_enrollment'],
    bins=[0, 20, 25, 30, 40, 100],
    labels=['<=20', '21-25', '26-30', '31-40', '>40']
)
df_dashboard['Gender_Label'] = df_dashboard['Gender'].map({1: 'Male', 0: 'Female'})
df_dashboard.to_csv('students_cleaned.csv', index=False)
print(f'Dashboard CSV saved: students_cleaned.csv ({len(df_dashboard)} rows)')

In [ ]:
# Untuk klasifikasi biner: hanya Dropout vs Graduate
# Excluded: Enrolled (masih aktif, belum ada outcome final)
df_binary = df[df['Status'].isin(['Dropout', 'Graduate'])].copy()
df_binary['Target'] = (df_binary['Status'] == 'Dropout').astype(int)

print(f'Dataset biner shape: {df_binary.shape}')
print(f'Class distribution:')
print(df_binary['Target'].value_counts())
print(f'Dropout rate: {df_binary["Target"].mean() * 100:.2f}%')

# Features
X = df_binary.drop(columns=['Status', 'Target'])
y = df_binary['Target']

print(f'\nFeatures: {X.shape[1]}')
print(f'Samples : {X.shape[0]}')

In [ ]:
# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Handle class imbalance dengan SMOTE pada training set saja
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Train set (sebelum SMOTE): {X_train.shape[0]} samples')
print(f'Train set (sesudah SMOTE) : {X_train_res.shape[0]} samples')
print(f'Test set                  : {X_test.shape[0]} samples')
print(f'\nResampled class distribution:\n{pd.Series(y_train_res).value_counts()}')

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

print('Feature scaling selesai.')

## Modeling

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, C=0.5
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=4,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=4,
        subsample=0.8, random_state=RANDOM_STATE
    )
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_scaled, y_train_res)

    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    acc      = accuracy_score(y_test, y_pred)
    roc_auc  = roc_auc_score(y_test, y_proba)
    f1       = f1_score(y_test, y_pred)
    cv_score = cross_val_score(model, X_train_scaled, y_train_res,
                               cv=cv, scoring='roc_auc').mean()

    results[name] = {
        'model': model,
        'accuracy': acc,
        'roc_auc': roc_auc,
        'f1_dropout': f1,
        'cv_roc_auc': cv_score,
        'y_pred': y_pred,
        'y_proba': y_proba
    }
    print(f'  Accuracy={acc:.4f}  ROC-AUC={roc_auc:.4f}  F1={f1:.4f}  CV-AUC={cv_score:.4f}')

print('\nSemua model selesai dilatih.')

## Evaluation

In [ ]:
# Summary table
summary = pd.DataFrame({
    name: {
        'Accuracy': f"{r['accuracy']:.4f}",
        'ROC-AUC': f"{r['roc_auc']:.4f}",
        'F1 (Dropout)': f"{r['f1_dropout']:.4f}",
        'CV ROC-AUC': f"{r['cv_roc_auc']:.4f}"
    }
    for name, r in results.items()
}).T

print('=== Perbandingan Model ===')
print(summary.to_string())

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 6))
colors_roc = ['#2196F3', '#FF9800', '#E91E63']

for (name, r), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    plt.plot(fpr, tpr, color=color, linewidth=2,
             label=f"{name} (AUC={r['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Semua Model', fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Graduate', 'Dropout'],
                yticklabels=['Graduate', 'Dropout'])
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Model terbaik: Gradient Boosting
best_model_name = max(results, key=lambda k: results[k]['roc_auc'])
best_result = results[best_model_name]

print(f'Model Terbaik: {best_model_name}')
print(f"  Accuracy     : {best_result['accuracy']:.4f}")
print(f"  ROC-AUC      : {best_result['roc_auc']:.4f}")
print(f"  F1 (Dropout) : {best_result['f1_dropout']:.4f}")
print()
print('Classification Report:')
print(classification_report(y_test, best_result['y_pred'],
                            target_names=['Graduate', 'Dropout']))

In [ ]:
# Feature Importance - Gradient Boosting
best_model = best_result['model']
feature_names = X.columns.tolist()
importances = best_model.feature_importances_

feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True).tail(15)

plt.figure(figsize=(10, 7))
plt.barh(feat_df['Feature'], feat_df['Importance'], color='#E91E63', edgecolor='white')
plt.xlabel('Feature Importance')
plt.title(f'Top 15 Feature Importances - {best_model_name}', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 10 Fitur Paling Penting:')
print(feat_df.tail(10)[['Feature', 'Importance']].sort_values('Importance', ascending=False).to_string())

## Deployment

Menyimpan model terbaik beserta scaler untuk digunakan pada `prediction.py` dan `app.py`.

In [ ]:
# Simpan model sebagai Pipeline (scaler + model)
os.makedirs('model', exist_ok=True)

final_pipeline = Pipeline([
    ('scaler', scaler),
    ('model', best_model)
])

joblib.dump(final_pipeline, 'model/best_model.pkl')
print(f'Model disimpan ke model/best_model.pkl')
print(f'Model: {best_model_name}')

In [ ]:
# Verifikasi: load dan test prediksi
loaded_model = joblib.load('model/best_model.pkl')

sample = X_test.iloc[:3]
preds  = loaded_model.predict(sample)
probas = loaded_model.predict_proba(sample)

print('=== Verifikasi Model ===' )
for i, (pred, prob) in enumerate(zip(preds, probas)):
    label = 'Dropout' if pred == 1 else 'Graduate'
    actual = 'Dropout' if y_test.iloc[i] == 1 else 'Graduate'
    print(f'  Sample {i+1}: Prediksi={label} | Aktual={actual} | Prob Dropout={prob[1]:.4f}')

print('\nModel berhasil diverifikasi.')

## Kesimpulan

Berdasarkan analisis dan pemodelan yang telah dilakukan:

1. **Dropout rate** Jaya Jaya Institut sekitar **32%** dari siswa yang memiliki outcome final (Dropout vs Graduate).

2. **Faktor paling berpengaruh** terhadap dropout:
   - Performa akademik semester 1 dan 2 (jumlah SKS lulus dan nilai rata-rata)
   - Status pembayaran uang kuliah
   - Usia saat pendaftaran
   - Status beasiswa
   - Status debtor

3. **Model Gradient Boosting** dipilih sebagai model terbaik karena memiliki ROC-AUC tertinggi dengan performa yang seimbang antara presisi dan recall untuk kelas Dropout.

4. **Rekomendasi**: Implementasikan sistem early warning berbasis model ini untuk monitoring siswa berisiko tinggi setiap akhir semester, terutama difokuskan pada siswa dengan nilai rendah di semester 1 dan yang memiliki masalah finansial.